In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/achiver7/arch3-model/transformers/default/1/run_arch3_swa_v2/latest.pt
/kaggle/input/models/achiver7/arch3-model/transformers/default/1/run_arch3_swa_v2/log.jsonl
/kaggle/input/models/achiver7/arch3-model/transformers/default/1/run_arch3_swa_v2/best.pt
/kaggle/input/datasets/achiver7/project-phase-1/data/sc_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/acts_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/acts_train.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/tribunals_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/test.json
/kaggle/input/datasets/achiver7/project-phase-1/data/sc_train.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/tribunals_train.jsonl
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/sc_train.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/acts_train.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/acts_val.bin
/kaggle

In [2]:
import os
import torch

CKPT = (
    "/kaggle/input/models/achiver7/"
    "arch3-model/transformers/default/1/"
    "run_arch3_swa_v2/latest.pt"
)

print("Exists:", os.path.exists(CKPT))

if os.path.exists(CKPT):
    print(
        "Size:",
        f"{os.path.getsize(CKPT) / (1024**3):.2f} GB"
    )

    ck = torch.load(
        CKPT,
        map_location="cpu",
        weights_only=False
    )

    print("Training step:", ck.get("step"))
    print("Best val:", ck.get("best"))
    print("Checkpoint W:", (ck.get("cfg") or {}).get("W"))
    print("Model tensors:", len(ck["model"]))

    del ck

Exists: True
Size: 2.71 GB
Training step: 10000
Best val: 2.3814771915475528
Checkpoint W: 256
Model tensors: 260


In [3]:
# ============================================================
# CS781 PHASE 1
# ARCHITECTURE 3 — SWA
# FULL TEST GENERATION + SUBMISSION
# ============================================================

import os
import json
import time
import math
import random
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from tokenizers import Tokenizer


# ============================================================
# 1. PATHS / SETTINGS
# ============================================================

MODE = "test"

# Generate ALL 1000 test prompts
RANGE = (0, 1000)

# Your actual trained checkpoint
CKPT = (
    "/kaggle/input/models/achiver7/"
    "arch3-model/transformers/default/1/"
    "run_arch3_swa_v2/latest.pt"
)

# Tokenizer + official test
TOKD = (
    "/kaggle/input/datasets/achiver7/"
    "phase1-tokens-v40k-clean1/tokens"
)

TEST_JSON = f"{TOKD}/test.json"
TOKENIZER_JSON = f"{TOKD}/tokenizer.json"

# Output MUST be /kaggle/working
OUT = "/kaggle/working/submission_arch3"

# Architecture
V = 40000
BLOCK = 1024
W = 256

DEVICE = "cuda"

# Prompt / generation
PROMPT_KEEP = 768
MAX_NEW = 256

# Decoding
TEMP = 0.9
TOP_K = 50
TOP_P = 1.0
SEED = 0

# If fewer than 128 words are generated
SHORT_POLICY = "resample"


# ============================================================
# 2. BASIC CHECKS
# ============================================================

assert torch.cuda.is_available(), (
    "CUDA GPU is not available. "
    "Turn on GPU in Kaggle."
)

assert os.path.exists(CKPT), (
    f"Checkpoint not found:\n{CKPT}"
)

assert os.path.exists(TEST_JSON), (
    f"Test file not found:\n{TEST_JSON}"
)

assert os.path.exists(TOKENIZER_JSON), (
    f"Tokenizer not found:\n{TOKENIZER_JSON}"
)

assert PROMPT_KEEP + MAX_NEW <= BLOCK


os.makedirs(
    OUT,
    exist_ok=True
)


print("=" * 70)
print("ARCHITECTURE 3 — TEST GENERATION")
print("=" * 70)

print("Checkpoint:")
print(CKPT)

print()
print("Output:")
print(OUT)

print()
print("Device:", DEVICE)

print()
print("Range:", RANGE)

print()


# ============================================================
# 3. POSITION EMBEDDING
# ============================================================

def sinusoidal_pos_emb(
    T,
    d_model,
    device
):

    pos = torch.arange(
        T,
        device=device,
        dtype=torch.float32
    ).unsqueeze(1)

    div = torch.exp(
        torch.arange(
            0,
            d_model,
            2,
            device=device,
            dtype=torch.float32
        )
        * (-math.log(10000.0) / d_model)
    )

    pe = torch.zeros(
        T,
        d_model,
        device=device,
        dtype=torch.float32
    )

    pe[:, 0::2] = torch.sin(
        pos * div
    )

    pe[:, 1::2] = torch.cos(
        pos * div
    )

    return pe.unsqueeze(0)


# ============================================================
# 4. SLIDING-WINDOW CAUSAL ATTENTION
# ============================================================

class SlidingWindowCausalSelfAttention(
    nn.Module
):

    def __init__(
        self,
        d_model,
        n_heads,
        block_size,
        window_size
    ):

        super().__init__()

        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.window_size = window_size

        self.q_proj = nn.Linear(
            d_model,
            d_model
        )

        self.k_proj = nn.Linear(
            d_model,
            d_model
        )

        self.v_proj = nn.Linear(
            d_model,
            d_model
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model
        )

        # ----------------------------------------------------
        # Exact Arch3 SWA mask
        #
        # token i can attend to token j iff:
        #
        # j <= i
        # i - j < W
        # ----------------------------------------------------

        idx = torch.arange(
            block_size
        )

        causal = (
            idx[None, :]
            <= idx[:, None]
        )

        window = (
            idx[:, None]
            - idx[None, :]
        ) < window_size

        mask = causal & window

        self.register_buffer(
            "swa_mask",
            mask.view(
                1,
                1,
                block_size,
                block_size
            ),
            persistent=False
        )


    def forward(self, x):

        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.n_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_heads,
            self.head_dim
        ).transpose(1, 2)

        attn_mask = (
            self.swa_mask[
                :,
                :,
                :T,
                :T
            ]
        )

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=0.0,
            is_causal=False
        )

        y = (
            y.transpose(1, 2)
            .contiguous()
            .view(B, T, C)
        )

        return self.out_proj(y)


# ============================================================
# 5. FEED FORWARD
# ============================================================

class FeedForward(
    nn.Module
):

    def __init__(
        self,
        d_model,
        d_ff
    ):

        super().__init__()

        self.fc1 = nn.Linear(
            d_model,
            d_ff
        )

        self.fc2 = nn.Linear(
            d_ff,
            d_model
        )


    def forward(self, x):

        return self.fc2(
            F.gelu(
                self.fc1(x)
            )
        )


# ============================================================
# 6. TRANSFORMER BLOCK
# ============================================================

class TransformerBlock(
    nn.Module
):

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        block_size,
        window_size
    ):

        super().__init__()

        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.attn = (
            SlidingWindowCausalSelfAttention(
                d_model=d_model,
                n_heads=n_heads,
                block_size=block_size,
                window_size=window_size
            )
        )

        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.mlp = FeedForward(
            d_model,
            d_ff
        )


    def forward(self, x):

        # Pre-LN residual block
        x = (
            x
            + self.attn(
                self.norm1(x)
            )
        )

        x = (
            x
            + self.mlp(
                self.norm2(x)
            )
        )

        return x


# ============================================================
# 7. LEGAL DECODER MODEL
# ============================================================

class LegalDecoderLM(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        block_size,
        d_model,
        n_layers,
        n_heads,
        d_ff,
        window_size
    ):

        super().__init__()

        self.vocab_size = vocab_size
        self.block_size = block_size
        self.d_model = d_model
        self.n_layers = n_layers

        # Token embedding
        self.token_emb = nn.Embedding(
            vocab_size,
            d_model
        )

        # sqrt(d_model) scaling
        self.embed_scale = math.sqrt(
            d_model
        )

        # Fixed sinusoidal position embedding
        pos = sinusoidal_pos_emb(
            block_size,
            d_model,
            device="cpu"
        )

        self.register_buffer(
            "pos_emb",
            pos,
            persistent=False
        )

        # 16 Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                block_size=block_size,
                window_size=window_size
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.LayerNorm(
            d_model
        )

        # Tied output head
        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = (
            self.token_emb.weight
        )

        # ----------------------------------------------------
        # Initialization
        # ----------------------------------------------------

        self.apply(
            self._init_weights
        )

        # GPT-style residual scaling
        for block in self.blocks:

            block.attn.out_proj.weight.data.mul_(
                1.0 / math.sqrt(
                    2 * self.n_layers
                )
            )

            block.mlp.fc2.weight.data.mul_(
                1.0 / math.sqrt(
                    2 * self.n_layers
                )
            )


    def _init_weights(
        self,
        module
    ):

        if isinstance(
            module,
            nn.Linear
        ):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:

                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(
            module,
            nn.Embedding
        ):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

        elif isinstance(
            module,
            nn.LayerNorm
        ):

            nn.init.ones_(
                module.weight
            )

            nn.init.zeros_(
                module.bias
            )


    def forward(self, x):

        B, T = x.shape

        assert T <= self.block_size, (
            f"Sequence length {T} "
            f"exceeds block_size "
            f"{self.block_size}"
        )

        tok = (
            self.token_emb(x)
            * self.embed_scale
        )

        pos = self.pos_emb[
            :,
            :T,
            :
        ].to(tok.dtype)

        h = tok + pos

        for block in self.blocks:

            h = block(h)

        h = self.final_norm(h)

        return self.lm_head(h)


# ============================================================
# 8. BUILD EXACT ARCHITECTURE 3
# ============================================================

def build_model(
    vocab_size,
    block_size,
    window_size=256
):

    d_model = 1024
    n_layers = 16
    n_heads = 16
    d_ff = 4096

    return LegalDecoderLM(
        vocab_size=vocab_size,
        block_size=block_size,
        d_model=d_model,
        n_layers=n_layers,
        n_heads=n_heads,
        d_ff=d_ff,
        window_size=window_size
    )


# ============================================================
# 9. LOAD TOKENIZER + TEST
# ============================================================

print("Loading tokenizer...")

tok = Tokenizer.from_file(
    TOKENIZER_JSON
)

print("Loading test...")

with open(
    TEST_JSON,
    "r",
    encoding="utf-8"
) as f:

    test = json.load(f)


print(
    f"Official test IDs: {len(test)}"
)


# ============================================================
# 10. BUILD MODEL
# ============================================================

print()
print("Building Architecture 3...")

model = build_model(
    V,
    BLOCK,
    W
).to(DEVICE)


# ============================================================
# 11. PARAMETER COUNT CHECK
# ============================================================

n_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"Parameters: {n_params:,}"
)

assert n_params == 242_501_632, (
    f"Unexpected parameter count: "
    f"{n_params:,}"
)

print(
    "Parameter count check: PASS"
)


# ============================================================
# 12. LOAD CHECKPOINT
# ============================================================

print()
print("Loading checkpoint...")

ck = torch.load(
    CKPT,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(
    ck["model"]
)

model.eval()

cfg_w = (
    (ck.get("cfg") or {})
    .get("W")
)

print()
print(
    f"Checkpoint: {CKPT}"
)

print(
    f"Training step: {ck.get('step')}"
)

print(
    f"Best validation loss: {ck.get('best')}"
)

print(
    f"Checkpoint W: {cfg_w}"
)

print(
    f"Script W: {W}"
)

assert cfg_w in (
    None,
    W
), (
    "Checkpoint window size "
    "does not match inference W"
)

del ck

print()
print("Checkpoint loaded successfully.")


# ============================================================
# 13. GENERATOR
# ============================================================

gen = torch.Generator(
    device=DEVICE
).manual_seed(
    SEED
)


# ============================================================
# 14. LAST-LOGITS FUNCTION
# ============================================================

@torch.no_grad()
def last_logits(x):

    T = x.size(1)

    h = (
        model.token_emb(x)
        * model.embed_scale
    )

    h = (
        h
        + model.pos_emb[
            :,
            :T,
            :
        ].to(h.dtype)
    )

    for block in model.blocks:

        h = block(h)

    # LayerNorm is position-wise,
    # so only last position is needed.
    h = model.final_norm(
        h[:, -1:, :]
    )

    return model.lm_head(
        h
    )[:, 0, :]


# ============================================================
# 15. VERIFY LAST-LOGITS OPTIMIZATION
# ============================================================

print()
print("Running startup equivalence check...")

with torch.no_grad():

    x_check = torch.randint(
        3,
        V,
        (1, 300),
        device=DEVICE
    )

    full_logits = (
        model(x_check)
        [:, -1, :]
        .float()
    )

    fast_logits = (
        last_logits(x_check)
        .float()
    )

    max_diff = (
        full_logits
        - fast_logits
    ).abs().max().item()

    print(
        f"Maximum difference: "
        f"{max_diff:.3e}"
    )

    assert torch.allclose(
        full_logits,
        fast_logits,
        atol=1e-3,
        rtol=1e-3
    ), (
        "last_logits does not match "
        "full forward"
    )

    del (
        x_check,
        full_logits,
        fast_logits
    )

print(
    "Startup check: PASS"
)


# ============================================================
# 16. SAMPLING
# ============================================================

def sample(logits):

    if TEMP == 0:

        return logits.argmax(
            -1,
            keepdim=True
        )

    logits = (
        logits / TEMP
    )

    # --------------------------------------------------------
    # Top-K
    # --------------------------------------------------------

    if TOP_K:

        values, _ = torch.topk(
            logits,
            min(
                TOP_K,
                logits.size(-1)
            )
        )

        logits[
            logits
            < values[:, [-1]]
        ] = -float("inf")

    # --------------------------------------------------------
    # Probability
    # --------------------------------------------------------

    probs = F.softmax(
        logits,
        dim=-1
    )

    # --------------------------------------------------------
    # Top-P
    # --------------------------------------------------------

    if TOP_P < 1.0:

        sorted_probs, sorted_indices = (
            torch.sort(
                probs,
                descending=True
            )
        )

        cumulative = (
            sorted_probs.cumsum(-1)
        )

        sorted_probs[
            (cumulative - sorted_probs)
            > TOP_P
        ] = 0

        probs = torch.zeros_like(
            probs
        ).scatter_(
            1,
            sorted_indices,
            sorted_probs
        )

        probs = (
            probs
            / probs.sum(
                -1,
                keepdim=True
            )
        )

    return torch.multinomial(
        probs,
        1,
        generator=gen
    )


# ============================================================
# 17. GENERATE ONE PROMPT
# ============================================================

@torch.no_grad()
def continue_text(prompt):

    # Tokenize complete prompt
    full_ids = tok.encode(
        prompt
    ).ids

    # Keep only final 768 tokens
    ids = full_ids[
        -PROMPT_KEEP:
    ]

    x = torch.tensor(
        [ids],
        device=DEVICE
    )

    generated = []

    while len(generated) < MAX_NEW:

        with torch.autocast(
            "cuda",
            dtype=torch.float16
        ):

            logits = last_logits(
                x
            ).float()

        nxt = sample(
            logits
        )

        x = torch.cat(
            [
                x,
                nxt
            ],
            dim=1
        )

        generated.append(
            nxt.item()
        )

        # Check every 8 tokens
        if (
            len(generated) % 8 == 0
            and len(
                tok.decode(
                    generated
                ).split()
            ) >= 128
        ):

            break

    words = tok.decode(
        generated
    ).split()

    return (
        words,
        len(full_ids)
    )


# ============================================================
# 18. GENERATE ALL PROMPTS
# ============================================================

def run_generation(
    prompts,
    tag
):

    part_path = (
        f"{OUT}/pred_"
        f"{tag}_"
        f"t{TEMP}_"
        f"k{TOP_K}_"
        f"p{TOP_P}_"
        f"s{SEED}.json"
    )

    meta_path = (
        f"{OUT}/meta_"
        f"{tag}_"
        f"t{TEMP}_"
        f"k{TOP_K}_"
        f"p{TOP_P}_"
        f"s{SEED}.json"
    )

    # --------------------------------------------------------
    # Resume if partially generated
    # --------------------------------------------------------

    if os.path.exists(
        part_path
    ):

        with open(
            part_path,
            "r",
            encoding="utf-8"
        ) as f:

            pred = json.load(f)

        print(
            f"Resuming existing file: "
            f"{part_path}"
        )

        print(
            f"Already generated: "
            f"{len(pred)}"
        )

    else:

        pred = {}


    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    if os.path.exists(
        meta_path
    ):

        with open(
            meta_path,
            "r",
            encoding="utf-8"
        ) as f:

            meta = json.load(f)

    else:

        meta = {
            "short_first_try": {},
            "resampled_ok": [],
            "src_tokens": {},
            "policy": SHORT_POLICY
        }


    todo = [
        k
        for k in prompts
        if k not in pred
    ]


    print()
    print(
        f"Total prompts: "
        f"{len(prompts)}"
    )

    print(
        f"Already done: "
        f"{len(pred)}"
    )

    print(
        f"Remaining: "
        f"{len(todo)}"
    )

    print()


    t0 = time.time()


    # --------------------------------------------------------
    # Generation loop
    # --------------------------------------------------------

    for n, key in enumerate(
        todo
    ):

        words, src_tokens = (
            continue_text(
                prompts[key]
            )
        )

        meta[
            "src_tokens"
        ][key] = src_tokens


        # ----------------------------------------------------
        # Handle short output
        # ----------------------------------------------------

        if len(words) < 128:

            meta[
                "short_first_try"
            ][key] = len(words)


            if SHORT_POLICY == "error":

                raise RuntimeError(
                    f"ID {key}: "
                    f"only {len(words)} "
                    f"words generated"
                )


            elif SHORT_POLICY == "resample":

                original_state = (
                    gen.get_state()
                )

                success = False

                for retry in range(
                    1,
                    21
                ):

                    gen.manual_seed(
                        1000 + retry
                    )

                    words, _ = (
                        continue_text(
                            prompts[key]
                        )
                    )

                    if len(words) >= 128:

                        meta[
                            "resampled_ok"
                        ].append(key)

                        success = True

                        break


                # Restore normal RNG stream
                gen.set_state(
                    original_state
                )


                if not success:

                    raise RuntimeError(
                        f"ID {key}: "
                        f"still short after "
                        f"20 resamples "
                        f"({len(words)} words)"
                    )


            elif SHORT_POLICY == "pad":

                assert words, (
                    f"ID {key}: "
                    "generated zero words"
                )

                words = (
                    words
                    * (
                        128 // len(words)
                        + 1
                    )
                )[:128]


        # ----------------------------------------------------
        # Store exactly 128 words
        # ----------------------------------------------------

        pred[key] = {
            "text": " ".join(
                words[:128]
            )
        }


        # ----------------------------------------------------
        # Save every 25 prompts
        # ----------------------------------------------------

        if (
            (n + 1) % 25 == 0
            or
            (n + 1) == len(todo)
        ):

            with open(
                part_path,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    pred,
                    f,
                    ensure_ascii=False
                )


            with open(
                meta_path,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    meta,
                    f,
                    ensure_ascii=False
                )


            elapsed = (
                time.time()
                - t0
            )

            per_prompt = (
                elapsed
                / (n + 1)
            )

            print(
                f"{len(pred)}/{len(prompts)} "
                f"done | "
                f"{per_prompt:.1f}s/prompt | "
                f"short first try: "
                f"{len(meta['short_first_try'])}"
            )


    # Final save
    with open(
        part_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            pred,
            f,
            ensure_ascii=False
        )


    with open(
        meta_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            meta,
            f,
            ensure_ascii=False
        )


    # --------------------------------------------------------
    # Final word-count check
    # --------------------------------------------------------

    bad = [
        k
        for k, v in pred.items()
        if len(
            v["text"].split()
        ) != 128
    ]

    assert not bad, (
        f"Outputs not exactly 128 words: "
        f"{bad[:10]}"
    )


    print()
    print(
        f"Generation complete: "
        f"{len(pred)} predictions"
    )

    print(
        f"Prediction file:\n"
        f"{part_path}"
    )

    return pred


# ============================================================
# 19. TEST GENERATION
# ============================================================

ids = list(
    test.keys()
)

start = RANGE[0]

end = (
    RANGE[1]
    if RANGE[1] is not None
    else len(ids)
)

assert 0 <= start < len(ids)

assert (
    start < end <= len(ids)
)


prompts = {
    key: test[key]["text"]
    for key in ids[start:end]
}


print()
print("=" * 70)
print("STARTING GENERATION")
print("=" * 70)

print(
    f"Generating IDs "
    f"{start} → {end}"
)

print(
    f"Number of prompts: "
    f"{len(prompts)}"
)

print()


pred = run_generation(
    prompts,
    f"test_{start}_{end}"
)


# ============================================================
# 20. CHECK PROMPT LEAKAGE
# ============================================================

print()
print("Checking prompt leakage...")

leaks = []

for key, value in pred.items():

    prompt_start = test[
        key
    ]["text"][:60]

    if prompt_start in value[
        "text"
    ]:

        leaks.append(key)


assert not leaks, (
    f"Prompt leakage found: "
    f"{leaks[:20]}"
)

print(
    "Prompt leakage: NONE"
)


# ============================================================
# 21. CHECK WORD COUNTS
# ============================================================

bad_lengths = []

for key, value in pred.items():

    n_words = len(
        value["text"].split()
    )

    if n_words != 128:

        bad_lengths.append(
            (key, n_words)
        )


assert not bad_lengths, (
    f"Bad word counts: "
    f"{bad_lengths[:20]}"
)

print(
    "128-word check: PASS"
)


# ============================================================
# 22. FULL SUBMISSION?
# ============================================================

if set(pred) == set(test):

    print()
    print("=" * 70)
    print("ALL 1000 TEST IDS GENERATED")
    print("=" * 70)


    # --------------------------------------------------------
    # prediction.json
    # --------------------------------------------------------

    prediction_path = (
        f"{OUT}/prediction.json"
    )

    with open(
        prediction_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            pred,
            f,
            ensure_ascii=False
        )


    # --------------------------------------------------------
    # submission.zip
    # --------------------------------------------------------

    zip_path = (
        f"{OUT}/submission.zip"
    )

    with zipfile.ZipFile(
        zip_path,
        "w",
        zipfile.ZIP_DEFLATED
    ) as z:

        z.write(
            prediction_path,
            arcname="prediction.json"
        )


    # --------------------------------------------------------
    # Verify ZIP
    # --------------------------------------------------------

    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as z:

        names = z.namelist()


    assert names == [
        "prediction.json"
    ]


    print()
    print(
        "prediction.json:"
    )

    print(
        prediction_path
    )

    print()
    print(
        "submission.zip:"
    )

    print(
        zip_path
    )

    print()
    print(
        "ZIP contents:"
    )

    print(
        names
    )

    print()
    print(
        "FINAL STATUS: READY FOR SUBMISSION"
    )


else:

    missing = (
        set(test)
        - set(pred)
    )

    extra = (
        set(pred)
        - set(test)
    )

    print()
    print("=" * 70)
    print("PARTIAL GENERATION")
    print("=" * 70)

    print(
        f"Generated: "
        f"{len(pred)}/{len(test)}"
    )

    print(
        f"Missing: "
        f"{len(missing)}"
    )

    print(
        f"Extra: "
        f"{len(extra)}"
    )

    print()
    print(
        "No submission.zip created."
    )

    print(
        "The generation JSON is saved and "
        "can be resumed."
    )

ARCHITECTURE 3 — TEST GENERATION
Checkpoint:
/kaggle/input/models/achiver7/arch3-model/transformers/default/1/run_arch3_swa_v2/latest.pt

Output:
/kaggle/working/submission_arch3

Device: cuda

Range: (0, 1000)

Loading tokenizer...
Loading test...
Official test IDs: 1000

Building Architecture 3...
Parameters: 242,501,632
Parameter count check: PASS

Loading checkpoint...

Checkpoint: /kaggle/input/models/achiver7/arch3-model/transformers/default/1/run_arch3_swa_v2/latest.pt
Training step: 10000
Best validation loss: 2.3814771915475528
Checkpoint W: 256
Script W: 256

Checkpoint loaded successfully.

Running startup equivalence check...
Maximum difference: 1.240e-05
Startup check: PASS

STARTING GENERATION
Generating IDs 0 → 1000
Number of prompts: 1000


Total prompts: 1000
Already done: 0
Remaining: 1000

25/1000 done | 4.5s/prompt | short first try: 0
50/1000 done | 4.6s/prompt | short first try: 0
75/1000 done | 4.6s/prompt | short first try: 0
100/1000 done | 4.6s/prompt | short 